# 实验二 C：基于 CNN 的人脸图像分类

**实验目的**

1. 以 LFW（Labelled Faces in the Wild，自然环境下的名人人脸数据集）为基础，完成人脸识别实验：
   输入是人脸图像，输出是人物身份标签；
2. 掌握图像数据预处理（裁剪、缩放、归一化）与多层卷积神经网络（3 层卷积）的结构设计与训练流程；
3. 通过损失曲线与准确率曲线，观察深度模型在视觉特征提取上的层次化表示能力。

**数据**：`lfw/` 目录，每个身份一个子文件夹，目录名即标签，图片是 250×250 的 RGB 人脸照片。
LFW 共有 5749 个身份、13233 张照片，多数身份只有一两张，直接做分类任务不现实，
因此本实验只保留照片数 ≥ 60 的 8 位人物（George W. Bush、Colin Powell、Tony Blair 等），
共 1348 张，读取方式与 `torchvision.datasets.ImageFolder` 的约定一致——
从 [天池 LFW](https://tianchi.aliyun.com/dataset/93615) 下载的数据解压后按同样的结构放入 `lfw/` 即可。

In [1]:
import time
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder

pio.templates.default = "plotly_white"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

C_BLUE, C_ORANGE, C_RED = "#2a78d6", "#eb6834", "#e34948"
C_INK, C_MUTED = "#0b0b0b", "#52514e"
BLUES = [[0.0, "#f2f7fe"], [0.35, "#cde2fb"], [0.7, "#6da7ec"], [1.0, "#2a78d6"]]

DATA_DIR = Path("lfw")     # 每个身份一个子文件夹，目录名即类别标签
if not DATA_DIR.exists():
    raise FileNotFoundError(
        "未找到 lfw/ 目录：请把 LFW 数据（每个身份一个子文件夹的图片）放在与本 notebook 相同的目录下，"
        "例如 lfw/George_W_Bush/George_W_Bush_0001.jpg")
print("torch", torch.__version__, "| 设备：", "cuda" if torch.cuda.is_available() else "cpu")

torch 2.13.0 | 设备： cpu


## 一、图像预处理

LFW 的图片已经裁剪到人脸区域（对齐后的人脸），但尺寸仍为 250×250、通道为 RGB，直接送进网络会有两个问题：
一是参数量和计算量偏大，二是同一个人的照片在光照、角度上有差异，需要统一到同一尺度。
因此本实验的预处理流程为：**缩放到 64×64 → 转灰度（人脸识别主要依赖结构信息，灰度可减少 2/3 的计算量）
→ 转张量并归一化**（`ToTensor` 把像素从 0–255 缩放到 0–1，再用整个数据集的均值方差做标准化）。

数据集按 8:2 随机划分为训练集与验证集；验证集不参与梯度更新，只用于观察模型是否过拟合。

In [2]:
tf = transforms.Compose([
    transforms.Resize((64, 64)),        # 统一尺寸
    transforms.Grayscale(num_output_channels=1),   # 转灰度
    transforms.ToTensor(),              # 归一化到 [0, 1]，并转成 (1, 64, 64)
    transforms.Normalize(mean=[0.5], std=[0.5]),   # 标准化到 [-1, 1]
])

full = ImageFolder(DATA_DIR, transform=tf)
class_names = full.classes
print(f"ImageFolder 读取到 {len(full)} 张图，{len(class_names)} 个身份类别")

x0, y0 = full[0]
print(f"单张图像张量形状：{tuple(x0.shape)}，取值范围 [{x0.min():.2f}, {x0.max():.2f}]，标签 {class_names[y0]}")

counts = np.bincount([label for _, label in full], minlength=len(class_names))
print("每个身份的样本数：")
for name, c in sorted(zip(class_names, counts), key=lambda t: -t[1]):
    print(f"  {name:<22} {c:>4} 张")
print(f"\n多数类基线（全部预测为样本最多的身份）准确率 = {counts.max() / counts.sum():.1%}，"
      f"随机猜测基线 = {1 / len(class_names):.1%}")

ImageFolder 读取到 1348 张图，8 个身份类别
单张图像张量形状：(1, 64, 64)，取值范围 [-1.00, 0.90]，标签 Ariel_Sharon
每个身份的样本数：
  George_W_Bush           530 张
  Colin_Powell            236 张
  Tony_Blair              144 张
  Donald_Rumsfeld         121 张
  Gerhard_Schroeder       109 张
  Ariel_Sharon             77 张
  Hugo_Chavez              71 张
  Junichiro_Koizumi        60 张

多数类基线（全部预测为样本最多的身份）准确率 = 39.3%，随机猜测基线 = 12.5%


In [3]:
# 按 8:2 划分训练集 / 验证集
g = torch.Generator().manual_seed(SEED)
idx = torch.randperm(len(full), generator=g).tolist()
n_train = int(len(full) * 0.8)
train_ds, val_ds = Subset(full, idx[:n_train]), Subset(full, idx[n_train:])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
print(f"训练集 {len(train_ds)} 张，验证集 {len(val_ds)} 张")

# 看几张图：预处理后的灰度人脸
fig = make_subplots(rows=1, cols=6, subplot_titles=[class_names[full[i][1]] for i in idx[:6]])
for k, i in enumerate(idx[:6], start=1):
    img = full[i][0][0].numpy()
    fig.add_trace(go.Heatmap(z=img, colorscale="Gray", showscale=False, hoverinfo="skip"),
                  row=1, col=k)
    # 图像热力图的纵轴必须反转，否则第一行像素会画在底部、整张脸是倒的
    fig.update_xaxes(visible=False, row=1, col=k)
    fig.update_yaxes(visible=False, autorange="reversed", row=1, col=k)

fig.update_layout(title=dict(text="预处理后的样本（64×64 灰度图）", x=0.02),
                  width=1120, height=260, margin=dict(l=30, r=30, t=60, b=20))
fig.show()

训练集 1078 张，验证集 270 张


## 二、卷积神经网络结构

网络由 3 个“卷积块”加 1 个全连接层组成，层次化地提取特征：

| 层 | 结构 | 输出尺寸 | 作用 |
|---|---|---|---|
| 卷积块 1 | Conv(1→16, 3×3) + ReLU + MaxPool(2) | 16×32×32 | 提取边缘、明暗等低级特征 |
| 卷积块 2 | Conv(16→32, 3×3) + ReLU + MaxPool(2) | 32×16×16 | 组合出眼睛、鼻子等局部结构 |
| 卷积块 3 | Conv(32→64, 3×3) + ReLU + MaxPool(2) | 64×8×8 | 形成人脸整体构型的高层特征 |
| 分类头 | Flatten → Linear(64×8×8→128) + ReLU → Linear(128→类别数) | 类别数 | 输出每个人的得分 |

池化层让特征图逐层变小、通道数逐层变多，感受野随之增大，这正是 CNN “从局部纹理到整体结构”的层次化表示过程。

In [4]:
class FaceCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


device = "cuda" if torch.cuda.is_available() else "cpu"
model = FaceCNN(len(class_names)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"运行设备：{device}")
print(f"模型参数量：{n_params:,}")
print(model)

运行设备：cpu
模型参数量：548,744
FaceCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=8, bias=True)
  )
)


## 三、训练过程

损失函数用交叉熵（多分类的标准选择），优化器用 Adam（学习率 1e-3），训练 20 轮，
每轮结束后在训练集与验证集上各算一次损失和准确率。

In [5]:
EPOCHS, LR = 20, 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(loader, train=False):
    """训练或评估一轮，返回 (平均损失, 准确率)"""
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    history["epoch"].append(epoch)
    for key, value in [("train_loss", tr_loss), ("val_loss", va_loss),
                       ("train_acc", tr_acc), ("val_acc", va_acc)]:
        history[key].append(value)
    if epoch % 4 == 0 or epoch == 1:
        print(f"epoch {epoch:>2} | 训练损失 {tr_loss:.4f} 准确率 {tr_acc:.4f} | "
              f"验证损失 {va_loss:.4f} 准确率 {va_acc:.4f}")
print(f"\n训练 {EPOCHS} 轮共耗时 {time.perf_counter() - t0:.1f} 秒")

epoch  1 | 训练损失 1.8211 准确率 0.3915 | 验证损失 1.7990 准确率 0.3593
epoch  4 | 训练损失 1.0251 准确率 0.6178 | 验证损失 0.9895 准确率 0.6407
epoch  8 | 训练损失 0.3186 准确率 0.8961 | 验证损失 0.8239 准确率 0.7741
epoch 12 | 训练损失 0.0524 准确率 0.9870 | 验证损失 0.8123 准确率 0.7963
epoch 16 | 训练损失 0.0036 准确率 1.0000 | 验证损失 0.7971 准确率 0.8333
epoch 20 | 训练损失 0.0014 准确率 1.0000 | 验证损失 0.8599 准确率 0.8370

训练 20 轮共耗时 35.5 秒


In [6]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("损失曲线", "准确率曲线"))
for key, name, color, col in [("train_loss", "训练损失", C_BLUE, 1), ("val_loss", "验证损失", C_ORANGE, 1),
                              ("train_acc", "训练准确率", C_BLUE, 2), ("val_acc", "验证准确率", C_ORANGE, 2)]:
    fig.add_trace(go.Scatter(
        x=history["epoch"], y=history[key], mode="lines+markers", name=name,
        line=dict(color=color, width=2), marker=dict(size=5), legendgroup=name,
        showlegend=(col == 2 or "loss" in key),
        hovertemplate=f"{name}<br>epoch %{{x}}：%{{y:.4f}}<extra></extra>",
    ), row=1, col=col)

fig.update_xaxes(title_text="epoch", row=1, col=1)
fig.update_xaxes(title_text="epoch", row=1, col=2)
fig.update_yaxes(title_text="交叉熵损失", row=1, col=1)
fig.update_yaxes(title_text="准确率", range=[0, 1.02], tickformat=".0%", row=1, col=2)
fig.update_layout(title=dict(text="训练过程：损失下降、准确率上升，随后趋于平缓", x=0.02),
                  width=1080, height=430, margin=dict(l=70, r=30, t=90, b=80),
                  legend=dict(orientation="h", yanchor="top", y=-0.2, x=0, title=None))
fig.show()

## 四、分类结果分析

In [7]:
# 验证集上的混淆矩阵与逐类准确率
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        y_pred += model(images.to(device)).argmax(1).cpu().tolist()
        y_true += labels.tolist()

acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"验证集整体准确率：{acc:.4f}（{len(y_true)} 张中判对 {int(acc * len(y_true))} 张）")

cm = confusion_matrix(y_true, y_pred)
per_class = cm.diagonal() / cm.sum(1)
print("\n各个身份的准确率：")
for name, a, n in sorted(zip(class_names, per_class, cm.sum(1)), key=lambda t: t[1]):
    print(f"  {name:<28} {a:.3f}（{n} 张）")

验证集整体准确率：0.8370（270 张中判对 226 张）

各个身份的准确率：
  Hugo_Chavez                  0.312（16 张）
  Gerhard_Schroeder            0.571（14 张）
  Junichiro_Koizumi            0.692（13 张）
  Donald_Rumsfeld              0.800（30 张）
  Tony_Blair                   0.806（31 张）
  Ariel_Sharon                 0.857（14 张）
  Colin_Powell                 0.873（55 张）
  George_W_Bush                0.979（97 张）


In [8]:
# 混淆矩阵（行=真实身份，列=预测身份）
fig = go.Figure(go.Heatmap(
    z=cm[::-1], x=class_names, y=class_names[::-1],
    colorscale=BLUES, zmin=0, zmax=cm.max(),
    text=cm[::-1], texttemplate="%{text}", textfont=dict(size=9, color=C_INK),
    colorbar=dict(title="样本数", thickness=13, len=0.9),
    hovertemplate="真实 %{y} → 预测 %{x}：%{z} 张<extra></extra>",
))
fig.update_layout(title=dict(text=f"验证集混淆矩阵（整体准确率 {acc:.1%}）", x=0.02))
fig.update_xaxes(title_text="预测身份", tickangle=45)
fig.update_yaxes(title_text="真实身份")
fig.update_layout(width=880, height=700, margin=dict(l=170, r=30, t=70, b=170))
fig.show()

In [9]:
# 随机抽 8 张验证集图片看预测结果，标题为“真实身份 → 预测身份”
# 先看预测，把“判对的”和“判错的”分开，各随机抽 4 张
correct, wrong = [], []
for i in range(len(val_ds)):
    img, label = val_ds[i]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device)).argmax(1).item()
    (correct if pred == label else wrong).append((img, label, pred))

pick = lambda pool, n: [pool[j] for j in torch.randperm(len(pool), generator=g)[:n].tolist()]
samples = pick(correct, 4) + pick(wrong, 4)

# 用 ✓ / ✗ 作为标题——不依赖颜色也能看出对错；判错的折成两行，避免标题互相重叠
titles = [f"✓ {class_names[label]}" if pred == label
          else f"✗ {class_names[label]} →<br>{class_names[pred]}"
          for _, label, pred in samples]

fig = make_subplots(rows=2, cols=4, horizontal_spacing=0.04, vertical_spacing=0.32,
                    subplot_titles=titles)
for k, (img, _, _) in enumerate(samples):
    fig.add_trace(go.Heatmap(z=img[0].numpy(), colorscale="Gray", showscale=False, hoverinfo="skip"),
                  row=k // 4 + 1, col=k % 4 + 1)
    fig.update_xaxes(visible=False, row=k // 4 + 1, col=k % 4 + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=k // 4 + 1, col=k % 4 + 1)

fig.update_layout(title=dict(text="验证集预测样例：上排判对、下排判错（✗ 给出真实身份 → 预测身份）", x=0.02),
                  width=1000, height=620, margin=dict(l=30, r=30, t=100, b=20))
fig.show()

## 五、实验小结

In [10]:
print("=" * 60)
print("实验结论（关键数值）")
print("=" * 60)
print(f"数据：{len(full)} 张人脸、{len(class_names)} 个身份；训练集 {len(train_ds)} 张、验证集 {len(val_ds)} 张")
print(f"预处理：250×250 RGB -> 64×64 灰度 -> 归一化到 [-1, 1]")
print(f"模型：3 层卷积（16/32/64 通道）+ 2 层全连接，参数量 {n_params:,}")
print("-" * 60)
print(f"训练 {EPOCHS} 轮：训练损失 {history['train_loss'][0]:.4f} -> {history['train_loss'][-1]:.4f}，"
      f"验证损失 {history['val_loss'][0]:.4f} -> {history['val_loss'][-1]:.4f}")
print(f"准确率：训练 {history['train_acc'][0]:.4f} -> {history['train_acc'][-1]:.4f}，"
      f"验证 {history['val_acc'][0]:.4f} -> {history['val_acc'][-1]:.4f}")
print(f"验证集最终准确率 {acc:.4f}，训练与验证准确率之差 {history['train_acc'][-1] - history['val_acc'][-1]:+.4f}")
print(f"参照基线：多数类 {counts.max() / counts.sum():.1%}，随机猜测 {1 / len(class_names):.1%}")
print(f"最难识别的身份：{class_names[int(np.argmin(per_class))]}（准确率 {per_class.min():.3f}，"
      f"样本数 {cm.sum(1)[int(np.argmin(per_class))]}）")
print(f"最容易识别的身份：{class_names[int(np.argmax(per_class))]}（准确率 {per_class.max():.3f}）")
print("=" * 60)

实验结论（关键数值）
数据：1348 张人脸、8 个身份；训练集 1078 张、验证集 270 张
预处理：250×250 RGB -> 64×64 灰度 -> 归一化到 [-1, 1]
模型：3 层卷积（16/32/64 通道）+ 2 层全连接，参数量 548,744
------------------------------------------------------------
训练 20 轮：训练损失 1.8211 -> 0.0014，验证损失 1.7990 -> 0.8599
准确率：训练 0.3915 -> 1.0000，验证 0.3593 -> 0.8370
验证集最终准确率 0.8370，训练与验证准确率之差 +0.1630
参照基线：多数类 39.3%，随机猜测 12.5%
最难识别的身份：Hugo_Chavez（准确率 0.312，样本数 16）
最容易识别的身份：George_W_Bush（准确率 0.979）
